# EEG Electrode Translator - Step 1 Notebook

This notebook starts with coordinates, visualization, one target location, and simple single-electrode ranking.

Use this first milestone before moving into pair or five-electrode montage optimization.

## 1. Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

plt.rcParams['figure.dpi'] = 140

## 2. Define or Load Electrode Coordinates

In Colab, upload `easycap_cac64_soterix_draft.csv` or replace `COORDINATE_FILE` with your own uploaded file path.

In [ ]:
COORDINATE_FILE = Path('easycap_cac64_soterix_draft.csv')

# If the file is not present in Colab, run this cell and upload it when prompted.
if not COORDINATE_FILE.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        COORDINATE_FILE = Path(next(iter(uploaded)))
    except Exception as exc:
        raise FileNotFoundError(
            'Upload easycap_cac64_soterix_draft.csv or set COORDINATE_FILE to the right path.'
        ) from exc

coords = pd.read_csv(COORDINATE_FILE)
coords['status'] = coords['status'].str.lower().str.strip()
coords.head()

## 3. Display Coordinate Table

In [ ]:
print(coords['status'].value_counts())
coords.sort_values(['status', 'y', 'x'], ascending=[True, False, True]).reset_index(drop=True)

## 4. Plot 2D Electrode Map

In [ ]:
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

def plot_2d_layout(df, title='2D electrode layout', target=None, highlight=None, save_path=None):
    highlight = set(highlight or [])
    fig, ax = plt.subplots(figsize=(8, 8))

    open_df = df[df['status'] == 'open']
    blocked_df = df[df['status'] == 'blocked']

    ax.scatter(blocked_df['x'], blocked_df['y'], s=260, c='#8fd19a', edgecolor='#2f6f3a', label='blocked EEG')
    ax.scatter(open_df['x'], open_df['y'], s=260, facecolor='white', edgecolor='#667085', linestyle='--', label='open Soterix')

    for row in df.itertuples():
        color = '#b42318' if row.label in highlight else '#111827'
        weight = 'bold' if row.label in highlight else 'normal'
        ax.text(row.x, row.y, row.label, ha='center', va='center', fontsize=7.5, color=color, fontweight=weight)

    if target is not None:
        ax.scatter([target['x']], [target['y']], s=220, marker='*', c='#f97316', edgecolor='black', label=target['name'])
        ax.text(target['x'], target['y'] + 0.25, target['name'], ha='center', fontsize=9, fontweight='bold')

    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_aspect('equal', adjustable='box')
    ax.grid(alpha=0.2)
    ax.legend(loc='upper right')

    if save_path:
        fig.savefig(save_path, bbox_inches='tight')
    return fig, ax

plot_2d_layout(coords, save_path=OUTPUT_DIR / 'electrode_layout_2D.png');

## 5. Plot 3D Electrode Map

This creates an approximate dome from the 2D cap-map coordinates. It is useful for orientation and quality control, but it is not a measured head model.

By default, the 3D plot does not label every electrode because full labeling becomes unreadable. Use `label_subset=[...]` to label only electrodes you are inspecting.

In [ ]:
def add_dome_z(df):
    out = df.copy()

    # Normalize the 2D drawing into a unit circle, preserving the cap map's
    # left/right and front/back proportions as much as possible.
    x_scale = np.abs(out['x']).max()
    y_scale = np.abs(out['y']).max()
    out['x3'] = out['x'] / x_scale
    out['y3'] = out['y'] / y_scale

    radius = np.sqrt(out['x3'] ** 2 + out['y3'] ** 2)
    radius = np.clip(radius, 0, 1)
    out['z3'] = np.sqrt(1 - radius ** 2)
    return out

coords_3d = add_dome_z(coords)

def plot_3d_layout(df, title='3D electrode layout', label_subset=None, save_path=None):
    label_subset = set(label_subset or [])
    fig = plt.figure(figsize=(8, 7))
    ax = fig.add_subplot(111, projection='3d')

    open_df = df[df['status'] == 'open']
    blocked_df = df[df['status'] == 'blocked']

    ax.scatter(blocked_df['x3'], blocked_df['y3'], blocked_df['z3'], s=70, c='#8fd19a', edgecolor='#2f6f3a', label='blocked EEG')
    ax.scatter(open_df['x3'], open_df['y3'], open_df['z3'], s=70, facecolor='white', edgecolor='#667085', label='open Soterix')

    for row in df[df['label'].isin(label_subset)].itertuples():
        ax.text(row.x3, row.y3, row.z3 + 0.04, row.label, ha='center', fontsize=8, fontweight='bold')

    # Add a faint head dome so the point cloud has context.
    u = np.linspace(0, 2 * np.pi, 40)
    v = np.linspace(0, np.pi / 2, 20)
    dome_x = np.outer(np.cos(u), np.sin(v))
    dome_y = np.outer(np.sin(u), np.sin(v))
    dome_z = np.outer(np.ones_like(u), np.cos(v))
    ax.plot_wireframe(dome_x, dome_y, dome_z, color='#d0d5dd', linewidth=0.35, alpha=0.35)

    ax.set_title(title)
    ax.set_xlabel('left/right')
    ax.set_ylabel('back/front')
    ax.set_zlabel('height')
    ax.set_xlim(-1.05, 1.05)
    ax.set_ylim(-1.05, 1.05)
    ax.set_zlim(0, 1.05)
    ax.set_box_aspect((1, 1, 0.7))
    ax.view_init(elev=28, azim=-62)
    ax.legend(loc='upper right')

    if save_path:
        fig.savefig(save_path, bbox_inches='tight')
    return fig, ax

plot_3d_layout(coords_3d, save_path=OUTPUT_DIR / 'electrode_layout_3D.png');

## 6. Define Target Location

Start with one target in the same coordinate system. You can later replace this with a region or imported anatomical target.

In [ ]:
target = {
    'name': 'example_target_left_frontal',
    'x': -1.6,
    'y': 2.8,
}

targets = pd.DataFrame([target])
targets.to_csv(OUTPUT_DIR / 'targets.csv', index=False)
targets

## 7. Calculate Distances to Target

In [ ]:
def distance_2d(x1, y1, x2, y2):
    return np.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2)

ranked = coords.copy()
ranked['distance_to_target'] = distance_2d(ranked['x'], ranked['y'], target['x'], target['y'])
ranked = ranked.sort_values('distance_to_target').reset_index(drop=True)
ranked.head(15)

## 8. Rank Single Open Electrodes

In [ ]:
ranked_single_electrodes = ranked[ranked['status'] == 'open'].copy()
ranked_single_electrodes.to_csv(OUTPUT_DIR / 'ranked_single_electrodes.csv', index=False)
ranked_single_electrodes.head(20)

## 9. Plot Top-Ranked Open Electrodes

In [ ]:
top_labels = ranked_single_electrodes.head(10)['label'].tolist()
plot_2d_layout(
    coords,
    title='Top open electrodes near target',
    target=target,
    highlight=top_labels,
    save_path=OUTPUT_DIR / 'top_single_electrodes_2D.png',
);

## 9b. Plot Top-Ranked Open Electrodes in 3D

This version labels only the top ranked open electrodes, which is much easier to read than labeling the whole cap.

In [ ]:
plot_3d_layout(
    coords_3d,
    title='Top open electrodes near target - 3D view',
    label_subset=top_labels,
    save_path=OUTPUT_DIR / 'top_single_electrodes_3D.png',
);

## 10. Save Outputs

This first notebook should create:

- `outputs/electrode_layout_2D.png`
- `outputs/electrode_layout_3D.png`
- `outputs/targets.csv`
- `outputs/ranked_single_electrodes.csv`
- `outputs/top_single_electrodes_2D.png`
- `outputs/top_single_electrodes_3D.png`